# Diffraction grating as a sum of phasors

*optics · intermediate* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/optics/diffraction_grating)

The far-field of $N$ slits is the sum of $N$ complex phasors, one per slit, each delayed by the extra path $d\sin\theta$. With complex tensors the whole pattern is one einsum. As $N$ grows from 2 to 20 the principal maxima stay put but get sharper, and $N-2$ faint secondary maxima appear between them.

**Vector form**

$$E(\theta) = \frac{1}{N}\sum_{n=0}^{N-1} e^{\,i k d n \sin\theta},\qquad I(\theta) = \operatorname{sinc}^2\!\Big(\frac{a\sin\theta}{\lambda}\Big)\,\lvert E(\theta)\rvert^2$$

**Index (tensor) form**

$$E_t = A_n\,P_{nt},\quad P_{nt} = e^{\,i k d\, n \sin\theta_t},\quad A_n = \tfrac{1}{N},\qquad I_t = S_t\,E_t^{*}E_t$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $n$ | `torch.arange(N)` | (N,) | slit number; slit $n$ is a distance $nd$ from slit 0 |
| $\sin\theta_t$ | `sin_theta` | (T,) | direction of observation, sampled on a fine grid |
| $P_{nt}$ | `phasors(n, sin_theta, k, d)` | (N, T) complex | phase factor of slit $n$ seen at angle $t$ |
| $E_t = A_n P_{nt}$ | `einsum('n,nt->t', A, P)` | (T,) complex | repeated $n$ ⇒ summed: add the waves of all slits |
| $S_t$ | `envelope(sin_theta, a, wavelength)` | (T,) | single-slit diffraction envelope $\operatorname{sinc}^2(a\sin\theta/\lambda)$ |
| $E^*_t E_t$ | `(E.conj() * E).real` | (T,) | intensity = squared magnitude of the complex amplitude |
| $\frac{\sin^2(N\phi/2)}{N^2\sin^2(\phi/2)}$ | `closed_form(sin_theta, N, k, d)` | (T,) | the same sum done by hand (geometric series), $\phi = kd\sin\theta$ |

The right panel draws the $N$ phasors head to tail at one angle. At a principal maximum ($d\sin\theta = m\lambda$) they all point the same way and add to length $N$. Moving off the maximum, the chain curls up and closes into a polygon when $N\phi = 2\pi$: the first zero, at an angle that shrinks like $1/N$. Intensities are normalised to 1 at the central peak; dotted lines mark $d\sin\theta = m\lambda$. Only the single-slit envelope depends on the slit width $a$.


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/optics/diffraction_grating/main.py
"""Main loop: parameters → tensors → solve → plot."""

from pathlib import Path

from problems._common.runtime import cli

from . import equation, plot, solve, tensors


def run(params: dict, out_dir: Path) -> Path:
    wavelength = params["wavelength"] * 1e-3  # nm → µm, so every length is in µm
    d, a = params["d"], params["a"]
    sin_theta = tensors.sin_theta_grid()
    counts = tensors.slit_counts(params["n_max"])

    patterns = solve.solve(counts, sin_theta, wavelength, d, a)
    frames = solve.frames(counts)
    orders = solve.principal_maxima(wavelength, d, sin_theta.max().item())

    print("principal maxima d sin θ = mλ at sin θ =", ", ".join(f"{s:+.3f}" for s in orders))
    first_zero = wavelength / (counts[-1] * d)
    print(f"first zero next to the central peak: sin θ = λ/(N d) = {first_zero:.4f} for N = {counts[-1]}")
    env = equation.envelope(sin_theta, a, wavelength)
    path = out_dir / "diffraction_grating.gif"
    return plot.animate(sin_theta, patterns, env, counts, frames, orders, wavelength, d, path)


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/optics/diffraction_grating/equation.py
r"""Physics → tensor translation.

Far from N equal slits (Fraunhofer limit) the wave from slit n, at height n d, travels an extra
path n d \sin\theta to reach direction θ, i.e. it picks up a phase k d n \sin\theta (k = 2π/λ).
The total complex amplitude is the sum of N unit phasors,

    E(\theta) = \frac{1}{N} \sum_{n=0}^{N-1} e^{\,i k d n \sin\theta}

In index notation with n = slit and t = sample of the angle:

    P_{nt} = e^{\,i k d\, n \sin\theta_t},    E_t = A_n P_{nt}        (repeated n ⇒ summed)

which is einsum('n,nt->t') on complex tensors. Each slit has a width a, which multiplies the
intensity by the single-slit envelope S_t = sinc²(a \sin\theta_t / λ):

    I_t = S_t\, E^*_t E_t

Summing the geometric series by hand gives the closed form (φ = k d \sin\theta)

    |E|^2 = \frac{\sin^2(N\phi/2)}{N^2 \sin^2(\phi/2)}
"""

import math

import torch


def phasors(n: torch.Tensor, sin_theta: torch.Tensor, k: float, d: float) -> torch.Tensor:
    r"""P_{nt} = exp(i k d n sin θ_t).   n: (N,), sin_theta: (T,) → (N, T) complex"""
    phase = k * d * torch.einsum("n,t->nt", n.to(sin_theta.dtype), sin_theta)  # (N, T) outer product
    return torch.exp(1j * phase)


def array_factor(n_slits: int, sin_theta: torch.Tensor, k: float, d: float) -> torch.Tensor:
    r"""E_t = A_n P_{nt}  with A_n = 1/N for every slit.   → (T,) complex"""
    n = torch.arange(n_slits)
    p = phasors(n, sin_theta, k, d)  # (N, T)
    amplitude = torch.full((n_slits,), 1.0 / n_slits, dtype=p.dtype)  # (N,) complex
    return torch.einsum("n,nt->t", amplitude, p)


def envelope(sin_theta: torch.Tensor, a: float, wavelength: float) -> torch.Tensor:
    """Single-slit factor S_t = sinc²(a sin θ / λ), with torch.sinc(x) = sin(πx)/(πx).   (T,) → (T,)"""
    return torch.sinc(a * sin_theta / wavelength) ** 2


def intensity(e: torch.Tensor, s: torch.Tensor) -> torch.Tensor:
    """I_t = S_t E*_t E_t.   (T,) complex, (T,) → (T,) real"""
    return s * (e.conj() * e).real


def closed_form(sin_theta: torch.Tensor, n_slits: int, k: float, d: float) -> torch.Tensor:
    """sin²(Nφ/2) / (N² sin²(φ/2)) with φ = k d sin θ; equal to 1 where sin(φ/2) = 0.   (T,) → (T,)"""
    half_phi = 0.5 * k * d * sin_theta
    num, den = torch.sin(n_slits * half_phi) ** 2, (n_slits * torch.sin(half_phi)) ** 2
    return torch.where(den < 1e-12, torch.ones_like(den), num / den.clamp(min=1e-30))


def wavenumber(wavelength: float) -> float:
    """k = 2π / λ."""
    return 2 * math.pi / wavelength


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/optics/diffraction_grating/tensors.py
"""Build the concrete tensors: the observation angles, the slit counts and the phasor angle shown in each frame."""

import math

import torch

FRAMES_PER_N = 6  # animation frames spent on each slit count


def sin_theta_grid(n_points: int = 4001) -> torch.Tensor:
    """Observation directions sin θ_t, fine enough to resolve the sharp peaks of 30 slits: (T,)."""
    return torch.linspace(-0.95, 0.95, n_points)


def slit_counts(n_max: int) -> list[int]:
    """N = 2, 3, …, n_max: one pattern per slit count."""
    return list(range(2, n_max + 1))


def phasor_angles(n_slits: int) -> torch.Tensor:
    """Phase step φ between neighbouring slits for the phasor panel, swept from the principal
    maximum (φ = 0) to the first zero (φ = 2π/N): shape (FRAMES_PER_N,)."""
    return torch.linspace(0.0, 2 * math.pi / n_slits, FRAMES_PER_N)


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/optics/diffraction_grating/solve.py
"""Compute the grating pattern for every slit count, and the phasor chains drawn in the animation."""

import torch

from .equation import array_factor, envelope, intensity, wavenumber
from .tensors import phasor_angles


def solve(counts: list[int], sin_theta: torch.Tensor, wavelength: float, d: float, a: float) -> torch.Tensor:
    """Intensity I_t for every N in counts: shape (len(counts), T), normalised to 1 at θ = 0."""
    k = wavenumber(wavelength)
    s = envelope(sin_theta, a, wavelength)  # (T,) the same for every N
    return torch.stack([intensity(array_factor(n, sin_theta, k, d), s) for n in counts])


def phasor_chain(n_slits: int, phi: float) -> torch.Tensor:
    """Tips of N unit phasors e^{i n φ} drawn head to tail, starting at 0: (N + 1,) complex.

    The last point is the (un-normalised) sum N·E; the chain closes into a polygon when Nφ = 2π.
    """
    steps = torch.exp(1j * phi * torch.arange(n_slits))  # (N,)
    return torch.cat([torch.zeros(1, dtype=steps.dtype), torch.cumsum(steps, dim=0)])


def frames(counts: list[int]) -> list[tuple[int, float, torch.Tensor]]:
    """One (index into counts, φ, phasor chain) entry per animation frame."""
    out = []
    for i, n in enumerate(counts):
        for phi in phasor_angles(n).tolist():
            out.append((i, phi, phasor_chain(n, phi)))
    return out


def principal_maxima(wavelength: float, d: float, sin_max: float) -> list[float]:
    """sin θ_m = m λ / d for every order m with |sin θ_m| ≤ sin_max."""
    m_max = int(sin_max * d / wavelength)
    return [m * wavelength / d for m in range(-m_max, m_max + 1)]


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/optics/diffraction_grating/plot.py
"""Animate the grating pattern as N grows, next to the chain of phasors at one angle."""

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

from problems._common.plotting import frame_indices, save_gif

from .equation import wavenumber


def animate(
    sin_theta: torch.Tensor,
    patterns: torch.Tensor,
    env: torch.Tensor,
    counts: list[int],
    frames: list,
    orders: list[float],
    wavelength: float,
    d: float,
    path: Path,
) -> Path:
    x = sin_theta.numpy()  # (T,)
    intensity = patterns.numpy()  # (len(counts), T)
    k = wavenumber(wavelength)

    fig, (ax, ax_p) = plt.subplots(1, 2, figsize=(7, 4), gridspec_kw={"width_ratios": [2, 1]})
    for s in orders:  # d sin θ = m λ
        ax.axvline(s, color="#9ca3af", lw=0.8, ls=":")
    ax.plot(x, env.numpy(), "--", color="#6b7280", lw=1, label="single-slit envelope")
    (curve,) = ax.plot([], [], color="#2563eb", lw=1.2)
    (marker,) = ax.plot([], [], "o", color="#dc2626", ms=5)
    ax.set_xlim(x[0], x[-1])
    ax.set_ylim(0, 1.05)
    ax.set_xlabel("sin θ")
    ax.set_ylabel("intensity I / I(0)")
    ax.legend(loc="upper right", fontsize=7)
    title = ax.set_title(" ")

    ax_p.set_aspect("equal")
    ax_p.set_xticks([])
    ax_p.set_yticks([])
    (chain,) = ax_p.plot([], [], "-o", color="#2563eb", ms=2.5, lw=1.2)
    (total,) = ax_p.plot([], [], color="#dc2626", lw=2)
    label = ax_p.set_title(" ")

    def update(f: int):
        i, phi, tips = frames[f]
        n = counts[i]
        curve.set_data(x, intensity[i])
        s = phi / (k * d)  # the angle this phase step belongs to: φ = k d sin θ
        marker.set_data([s], [np.interp(s, x, intensity[i])])
        title.set_text(f"N = {n} slits")
        z = tips.numpy()
        chain.set_data(z.real, z.imag)
        total.set_data([0, z[-1].real], [0, z[-1].imag])  # the resultant N·E
        ax_p.set_xlim(-0.3 * n, 1.1 * n)
        ax_p.set_ylim(-0.35 * n, 0.6 * n)
        label.set_text(f"phasors, Nφ = {n * phi / np.pi:.1f}π")

    fig.tight_layout()
    return save_gif(fig, update, frame_indices(len(frames)), path, fps=8)


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'wavelength': 500.0,
    'd': 2.0,
    'a': 0.5,
    'n_max': 20
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.optics.diffraction_grating")]:
    del sys.modules[name]
main = importlib.import_module("problems.optics.diffraction_grating.main")
problem_dir = Path("problems/optics/diffraction_grating")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
